# Task 3 — OOF amount disagreement

Read-only comparison of saved out-of-fold predictions. No fitting and no Test labels.


In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.pipeline.random_seed import set_seed
SEED = set_seed()
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import fbeta_score, mean_absolute_error
SRC = ROOT / "src"


## Reconstruct four saved procedures
The 80/20 formula and 0.50 gate are fixed from the model notebooks.


In [2]:
TASK = SRC / "task3"; RES = TASK / "models" / "results"
card = json.loads((TASK / "data" / "eda" / "eda_combined_v1.json").read_text(encoding="utf-8"))
train = pd.read_parquet(SRC / card["splits"]["train"]["path"])
from sklearn.model_selection import train_test_split
y = train.spending_30d.astype(float)
bands = pd.cut(y, bins=[-1,0,100,1000,10000,50000,np.inf], labels=False)
dev_idx, hold_idx = train_test_split(np.arange(len(y)), test_size=.2, stratify=bands, random_state=SEED)
dev = train.iloc[dev_idx].reset_index(drop=True)
parts=[]
for i in range(1,6):
 gate=pd.read_parquet(RES/f"round3_gate_fold{i}_v1.parquet")
 blend=pd.read_parquet(RES/f"round3_blend_fold{i}_v1.parquet")
 log=pd.read_parquet(RES/f"t3_log_positive_amount_cv5_v5-fold{i}.parquet")
 rmse=pd.read_parquet(RES/f"t3_log_rmse_amount_cv5_v7-fold{i}.parquet")
 assert gate.row_pos.equals(blend.row_pos) and gate.row_pos.equals(log.row_pos) and gate.row_pos.equals(rmse.row_pos)
 assert gate.true.equals(blend.true)
 d=gate[["fold","row_pos","true","probability","positive_amount"]].copy()
 d["continuous"]=np.maximum(.4*blend.cat+.6*blend.lgb,0)
 d["baseline"] = .8*np.where(d.probability>=.5,np.maximum(d.positive_amount,0),0)+.2*d.continuous
 d["log_mae"] = .8*np.where(d.probability>=.5,log.amount,0)+.2*d.continuous
 d["log_rmse"] = .8*np.where(d.probability>=.5,rmse.amount,0)+.2*d.continuous
 d["half_blend"] = (d.log_mae+d.log_rmse)/2
 parts.append(d)
oof=pd.concat(parts,ignore_index=True).sort_values("row_pos").reset_index(drop=True)
assert np.array_equal(oof.row_pos,np.arange(len(dev))) and np.array_equal(oof.true,dev.spending_30d)
names=["baseline","log_mae","log_rmse","half_blend"]
scores=pd.DataFrame({name:oof.groupby("fold").apply(lambda g:mean_absolute_error(g.true,g[name]),include_groups=False) for name in names})
print("Mean fold MAE in THB; the competition NMAE denominator is not inferred here.")
display(scores.agg(["mean","std"]).T.sort_values("mean").round(2))
print("True distribution: zero",round((oof.true==0).mean()*100,2),"%; >=50k",round((oof.true>=50000).mean()*100,2),"%")


Mean fold MAE in THB; the competition NMAE denominator is not inferred here.


,mean,std
half_blend,2811.16,81.40
log_mae,2814.97,82.33
log_rmse,2833.45,80.24
baseline,2895.92,83.67


True distribution: zero 48.12 %; >=50k 8.16 %


## Amount gaps and error tradeoffs
Threshold counts describe predicted spending amounts, not class labels.


In [3]:
def pair(a,b):
 z=oof[["true",a,b]].rename(columns={a:"a",b:"b"}).copy()
 z["abs_a"]=(z.a-z.true).abs();z["abs_b"]=(z.b-z.true).abs()
 z["delta"] = z.b-z.a
 z["b_improves"] = z.abs_b < z.abs_a
 return z
out=[]
for a,b in [("baseline","log_mae"),("log_mae","log_rmse"),("log_mae","half_blend")]:
 z=pair(a,b)
 for group,mask in [("all",np.ones(len(z),bool)),("zero",z.true==0),
                    ("positive",z.true>0),("tail_50k",z.true>=50000)]:
  q=z.loc[mask]
  out.append({"a":a,"b":b,"group":group,"rows":len(q),"MAE_a":q.abs_a.mean(),
              "MAE_b":q.abs_b.mean(),"mean_abs_prediction_gap":q.delta.abs().mean(),
              "gap_ge_1000_pct":100*(q.delta.abs()>=1000).mean(),
              "gap_ge_5000_pct":100*(q.delta.abs()>=5000).mean(),
              "b_improves_pct":100*q.b_improves.mean()})
display(pd.DataFrame(out).round(2))
for name in names:
 print(name,"pred >=1k",int((oof[name]>=1000).sum()),"pred >=10k",int((oof[name]>=10000).sum()),
       "pred >=50k",int((oof[name]>=50000).sum()))
for cutoff in (1000,10000,50000):
 a=oof.log_mae>=cutoff; b=oof.half_blend>=cutoff
 print("log-MAE vs half-blend crossing",cutoff,"THB:",int((a!=b).sum()),
       "up",int((~a&b).sum()),"down",int((a&~b).sum()))


,a,b,group,rows,MAE_a,MAE_b,mean_abs_prediction_gap,gap_ge_1000_pct,gap_ge_5000_pct,b_improves_pct
0,baseline,log_mae,all,83200,2895.92,2814.97,488.58,11.11,2.31,26.63
1,baseline,log_mae,zero,40039,1927.89,1923.42,93.85,1.86,0.29,13.60
2,baseline,log_mae,positive,43161,3793.93,3642.03,854.76,19.70,4.18,38.72
3,baseline,log_mae,tail_50k,6786,18923.14,18294.04,3829.39,79.06,24.43,53.99
4,log_mae,log_rmse,all,83200,2814.97,2833.45,402.28,10.22,1.64,24.77
5,log_mae,log_rmse,zero,40039,1923.42,1941.42,77.86,1.56,0.24,13.80
6,log_mae,log_rmse,positive,43161,3642.03,3660.94,703.23,18.27,2.94,34.95
7,log_mae,log_rmse,tail_50k,6786,18294.04,18146.97,3000.29,74.49,16.34,51.86
8,log_mae,half_blend,all,83200,2814.97,2811.16,201.14,6.49,0.35,25.53
9,log_mae,half_blend,zero,40039,1923.42,1932.42,38.93,0.98,0.05,13.80


baseline pred >=1k 36942 pred >=10k 12180 pred >=50k 7263
log_mae pred >=1k 36957 pred >=10k 12091 pred >=50k 7168
log_rmse pred >=1k 37057 pred >=10k 12142 pred >=50k 7342
half_blend pred >=1k 37007 pred >=10k 12123 pred >=50k 7255
log-MAE vs half-blend crossing 1000 THB: 144 up 97 down 47
log-MAE vs half-blend crossing 10000 THB: 68 up 50 down 18
log-MAE vs half-blend crossing 50000 THB: 125 up 106 down 19


## Distribution of close-model differences
These slices are descriptive, with no fitting or threshold selection.


In [4]:
z=pair("log_mae","half_blend").join(dev.drop(columns="spending_30d"),how="left")
for feature in ["prev_month_spending","historical_spending","days_since_last_purchase","purchase_frequency"]:
 bucket=pd.qcut(z[feature],5,duplicates="drop").astype(str).replace("nan","missing")
 p=z.assign(bucket=bucket).groupby("bucket",dropna=False).agg(rows=("delta","size"),
  mean_abs_gap=("delta",lambda s:s.abs().mean()),mae_log=("abs_a","mean"),
  mae_half=("abs_b","mean"),half_better=("b_improves","mean"))
 p["mae_change"]=p.mae_half-p.mae_log
 print("\n",feature);display(p.round(2))
print("Largest individual amount differences, with true amounts:")
display(z.assign(abs_gap=z.delta.abs()).nlargest(12,"abs_gap")[["true","a","b","delta","abs_a","abs_b","prev_month_spending","historical_spending"]].round(2))



 prev_month_spending


,rows,mean_abs_gap,mae_log,mae_half,half_better,mae_change
bucket,,,,,,
"(0.00692, 175.309]",15673,0.46,39.40,39.35,0.04,-0.06
"(175.309, 367.796]",15672,1.64,135.61,135.48,0.06,-0.13
"(1823.453, 63063.626]",15673,892.07,12096.78,12093.39,0.50,-3.39
"(367.796, 836.595]",15673,12.12,460.27,460.04,0.20,-0.23
"(836.595, 1823.453]",15672,50.84,1181.03,1178.28,0.48,-2.75
missing,4837,358.46,3338.14,3293.92,0.25,-44.22



 historical_spending


,rows,mean_abs_gap,mae_log,mae_half,half_better,mae_change
bucket,,,,,,
"(0.0249, 506.537]",15674,0.45,39.43,39.38,0.04,-0.05
"(1050.5, 2412.215]",15674,12.29,461.98,461.89,0.20,-0.08
"(2412.215, 5204.923]",15674,50.17,1150.16,1146.99,0.48,-3.17
"(506.537, 1050.5]",15674,1.66,120.34,119.99,0.06,-0.35
"(5204.923, 163870.532]",15674,870.90,12068.76,12081.43,0.49,12.67
missing,4830,429.06,3574.91,3480.09,0.26,-94.82



 days_since_last_purchase


,rows,mean_abs_gap,mae_log,mae_half,half_better,mae_change
bucket,,,,,,
"(-0.0005070000000000001, 12.193]",15649,314.19,4175.27,4174.64,0.36,-0.63
"(12.193, 24.139]",15649,253.71,3608.35,3605.22,0.31,-3.13
"(24.139, 36.158]",15648,203.69,2874.18,2868.39,0.23,-5.79
"(36.158, 50.899]",15649,147.31,2158.92,2150.56,0.21,-8.36
"(50.899, 139.116]",15649,87.11,1300.80,1303.52,0.17,2.72
missing,4956,200.10,2680.21,2664.29,0.24,-15.93



 purchase_frequency


,rows,mean_abs_gap,mae_log,mae_half,half_better,mae_change
bucket,,,,,,
"(0.999, 5.52]",15678,105.76,1543.53,1540.84,0.19,-2.69
"(11.349, 14.464]",15678,228.98,3179.61,3175.40,0.27,-4.21
"(14.464, 25.0]",15678,302.19,4056.60,4052.01,0.34,-4.59
"(5.52, 7.822]",15678,155.10,2212.12,2202.21,0.22,-9.90
"(7.822, 11.349]",15678,213.16,3111.64,3113.07,0.25,1.42
missing,4810,202.75,2721.53,2720.80,0.26,-0.73


Largest individual amount differences, with true amounts:


,true,a,b,delta,abs_a,abs_b,prev_month_spending,historical_spending
18182,94951.31,13647.90,33905.62,20257.72,81303.41,61045.69,NaN,NaN
61790,104178.47,15377.65,34971.98,19594.33,88800.82,69206.49,NaN,NaN
54342,106793.31,15778.88,32324.73,16545.84,91014.43,74468.58,NaN,NaN
82723,133100.35,15385.29,31441.11,16055.81,117715.06,101659.24,NaN,NaN
8956,245449.19,247143.36,232653.11,-14490.25,1694.17,12796.08,31499.73,83861.85
29088,94034.81,12660.45,26996.78,14336.33,81374.36,67038.03,NaN,NaN
61903,231891.70,180677.94,193921.17,13243.23,51213.76,37970.53,NaN,113079.92
82297,168416.85,219252.26,206101.02,-13151.25,50835.41,37684.17,23377.07,61422.56
52993,52113.74,12340.78,24916.55,12575.77,39772.96,27197.19,NaN,NaN
69025,135056.32,20154.04,32337.61,12183.57,114902.28,102718.71,NaN,NaN


## Assessment

- **Medium priority tail backup:** the fixed half log-MAE/log-RMSE blend. Its overall MAE gain is only 3.81 THB, but tail MAE improves 170.46 THB; zero-target MAE worsens 9 THB.
- **Low priority:** standalone log-RMSE loses overall on all five paired folds.
- The 125 crossings of predicted 50,000 THB measure amount distribution differences, not a classification score. Competition NMAE is not estimated here.
